# 神經網路

你在烘咖啡。每一爐有兩個數：溫度（°C）和時間（分鐘）。結果只有烘得好或烘不好。好烘大致落在 175–260°C、12–15 分鐘這一塊，而且時間不能比溫度允許的上限更長。

這一份用一個小神經網路學這塊區域。網路有兩層 sigmoid：輸入 2 個特徵，第一層 3 個單元，第二層 1 個單元。程式已經寫好。請 agent 改的格子，標題會寫「實驗」。下面這種格子只有你能寫，agent 不要填、不要改寫：


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>

</div>


## 給 agent

- 可以改標題含「實驗」的程式格，也可以依學生當下的問題加新的程式格。
- 不要改標題是「你的觀察」的格子，也不要替學生寫結論。
- 不要改 1.7 的學習率、epoch、複製份數，也不要拿掉 1.3 存好的權重，除非學生明確說要比較這一次訓練的結果。
- 學生若只說「幫我做完」，先問他想把門檻調高還是調低，再改程式。

## 這一爐烘得好不好

![烘焙師看著溫度和時間，判斷這一爐咖啡烘得好不好](images/roasting.jpg)

每一爐記下溫度和時間，結果是好烘（`y = 1`）或不好（`y = 0`）。例如 `200°C、13.9 分鐘` 落在好烘區裡，`200°C、17 分鐘` 時間太長，是不好。

這一份最後要做出的，就是下面這條流程：把一爐的溫度和時間先正規化，經過第一層 3 個 sigmoid 單元，再由第二層 1 個 sigmoid 算出好烘機會，最後跟門檻比。圖上的數字用的是 1.3 存好的那組權重。下面一格一格拆開來做。

![一爐咖啡從溫度、時間到預測結果的流程](images/flow.png)

這一份的順序：

- **1.1** 資料：好烘落在哪一塊
- **1.2** 正規化：把溫度和時間拉到差不多大
- **1.3** 一個單元：加權相加再過 sigmoid
- **1.4** 一層：3 個單元一起算
- **1.5** 前向傳播：兩層接起來
- **1.6** 用 Keras 搭同一個網路
- **1.7** 訓練：讓損失往下降
- **1.8** 每個單元管哪一塊
- **1.9** 實驗：改門檻

### 1.1 資料：好烘落在哪一塊

沒有資料檔。下一格用固定種子造 200 筆，所以每次圖一樣。紅色叉是好烘，藍色圈是不好。紫線標出造資料時用的那塊好烘區域，不是模型畫的。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def load_coffee_data():
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1, 2)
    X[:, 1] = X[:, 1] * 4 + 11.5
    X[:, 0] = X[:, 0] * (285 - 150) + 150
    Y = np.zeros(len(X))
    for i, (temperature, duration) in enumerate(X):
        limit = -3 / (260 - 175) * temperature + 21
        good = (
            175 < temperature < 260
            and 12 < duration < 15
            and duration <= limit
        )
        Y[i] = 1 if good else 0
    return X, Y.reshape(-1, 1)

def draw_good_region(ax):
    temperature = np.linspace(175, 260, 50)
    ax.plot(temperature, (-3 / 85) * temperature + 21, color="#7030A0", linewidth=1)
    ax.axhline(12, color="#7030A0", linewidth=1)
    ax.axvline(175, color="#7030A0", linewidth=1)

X, Y = load_coffee_data()
print("形狀", X.shape, Y.shape)
print("溫度最小/最大", round(float(X[:, 0].min()), 1), round(float(X[:, 0].max()), 1))
print("時間最小/最大", round(float(X[:, 1].min()), 1), round(float(X[:, 1].max()), 1))
print("好烘筆數", int(Y.sum()))

good = Y.reshape(-1) == 1
fig, ax = plt.subplots()
ax.scatter(X[good, 0], X[good, 1], marker="x", c="red", s=70, label="good")
ax.scatter(X[~good, 0], X[~good, 1], marker="o", facecolors="none", edgecolors="#0D5BDC", s=80, label="bad")
draw_good_region(ax)
ax.set_xlabel("Temperature (Celsius)")
ax.set_ylabel("Duration (minutes)")
ax.legend(loc="upper right")
plt.show()

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">好烘是不是集中在某一塊？</li>
<li style="color:#1a1a1a;">溫度太低或時間太長時，看起來怎樣？</li>
</ol>

</div>


### 1.2 正規化：把溫度和時間拉到差不多大

溫度是一百多度，時間是十幾分鐘。尺度差這麼多時，同一步長會被溫度拉著走。每個特徵減掉自己的平均，再除以自己的標準差：

$$
x_{\text{norm}} = \frac{x - \mu}{\sigma}
$$

這裡用 Keras 的 `Normalization` 做這件事。`adapt` 從這 200 筆算出平均和標準差，之後預測新的一爐，也要用同一組，不能重算。下一格左右兩張圖是同一批點，只有座標軸的刻度變了。

In [ ]:
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)
Xn = np.asarray(norm_l(X))
print("平均   ", np.asarray(norm_l.mean).reshape(-1).round(2))
print("標準差 ", np.sqrt(np.asarray(norm_l.variance)).reshape(-1).round(2))
print("正規化後 溫度 最小/最大", Xn[:, 0].min().round(2), Xn[:, 0].max().round(2))
print("正規化後 時間 最小/最大", Xn[:, 1].min().round(2), Xn[:, 1].max().round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in [(axes[0], X, "before"), (axes[1], Xn, "after normalization")]:
    ax.scatter(data[good, 0], data[good, 1], marker="x", c="red", s=30)
    ax.scatter(data[~good, 0], data[~good, 1], marker="o", facecolors="none", edgecolors="#0D5BDC", s=30)
    ax.set_title(title)
    ax.set_xlabel("Temperature")
axes[0].set_ylabel("Duration")
plt.tight_layout()
plt.show()

### 1.3 一個單元：加權相加再過 sigmoid

神經網路裡的一個單元，做的事和邏輯回歸一模一樣：把輸入加權相加，再過 sigmoid。

$$
a = g(w_1 x_1 + w_2 x_2 + b) \qquad g(z) = \frac{1}{1+e^{-z}}
$$

先借用一組存好的權重，它是在正規化之後的特徵上訓練出來的。`W1` 的每一直行是第一層一個單元的 `w`，`b1` 是三個單元各自的 `b`。下一格拿 `200°C、13.9 分鐘` 這一爐，只算第一層的單元 0。

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

W1 = np.array(
    [
        [-8.94, 0.29, 12.89],
        [-0.17, -7.34, 10.79],
    ]
)
b1 = np.array([-9.87, -9.28, 1.01])
W2 = np.array([[-31.38], [-27.86], [-32.79]])
b2 = np.array([15.54])

x = np.array([[200, 13.9]])
xn = np.asarray(norm_l(x))[0]
print("這一爐", x[0], "正規化後", xn.round(2))

w = W1[:, 0]
z = w[0] * xn[0] + w[1] * xn[1] + b1[0]
print(f"z = ({w[0]})({xn[0]:.2f}) + ({w[1]})({xn[1]:.2f}) + ({b1[0]}) = {z:.2f}")
print(f"單元 0 的輸出 a = g({z:.2f}) = {sigmoid(z):.3f}")

### 1.4 一層：3 個單元一起算

第一層有 3 個單元，每個都做 1.3 那件事，只是用各自那一直行的 `w`。三個一起算，就是一次矩陣乘法：

$$
\vec{a}^{[1]} = g\left(\vec{x}\,W^{[1]} + \vec{b}^{[1]}\right)
$$

`W1` 是 2×3：2 個輸入、3 個單元。下一格寫成 `dense`，單元 0 應該和 1.3 算出來的一樣。

In [ ]:
def dense(a_in, W, b):
    return sigmoid(a_in @ W + b)

print("W1 形狀", W1.shape, " b1 形狀", b1.shape)
a1 = dense(xn, W1, b1)
for unit, value in enumerate(a1):
    print(f"單元 {unit} 的輸出 {value:.3f}")

### 1.5 前向傳播：兩層接起來

第一層的 3 個輸出，就是第二層的輸入。第二層只有 1 個單元，輸出就是「好烘」的機會：

$$
a^{[2]} = g\left(\vec{a}^{[1]}\,W^{[2]} + b^{[2]}\right)
$$

從輸入一層一層往後算到輸出，叫前向傳播。下一格同時算兩爐：`200°C、13.9 分鐘`（落在好烘區裡）和 `200°C、17 分鐘`（時間太長）。

In [ ]:
def forward(x_norm):
    a1 = dense(x_norm, W1, b1)
    a2 = dense(a1, W2, b2)
    return a1, a2

X_try = np.array([[200, 13.9], [200, 17]])
a1_try, a2_try = forward(np.asarray(norm_l(X_try)))
for row, hidden, prob in zip(X_try, a1_try, a2_try):
    print(f"{row[0]:.0f}°C、{row[1]} 分鐘 → 第一層 {hidden.round(3)} → 好烘機會 {prob[0]:.3f}")

### 1.6 用 Keras 搭同一個網路

1.3 到 1.5 用 numpy 自己算，Keras 的 `Dense` 做的是同一件事。下一格搭出一樣的兩層，`summary` 會列出每層有幾個參數：第一層 `2×3 + 3 = 9`，第二層 `3×1 + 1 = 4`。

把存好的 `W1`、`b1`、`W2`、`b2` 放進去之後，Keras 算出的機會應該和 1.5 一樣。

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

tf.random.set_seed(1234)

def build_model():
    return Sequential(
        [
            tf.keras.Input(shape=(2,)),
            Dense(3, activation="sigmoid", name="layer1"),
            Dense(1, activation="sigmoid", name="layer2"),
        ]
    )

model = build_model()
model.summary()

model.get_layer("layer1").set_weights([W1, b1])
model.get_layer("layer2").set_weights([W2, b2])
keras_prob = model.predict(norm_l(X_try), verbose=0).reshape(-1)
print("Keras 算的好烘機會", keras_prob.round(3))
print("1.5 自己算的      ", a2_try.reshape(-1).round(3))

### 1.7 訓練：讓損失往下降

上面的權重是借來的。這一格從隨機的權重開始，自己訓練一個新的網路 `trained`。損失是二元交叉熵，就是邏輯回歸用的那個 loss；優化器是 Adam，梯度下降的一種，會自己調每一步的大小。

訓練把這 200 筆複製 1000 份，讓每個 epoch 多看幾次同樣的例子。學習率 `0.01`，跑 10 個 epoch。這些數字寫死，不在這一格改。

In [ ]:
Xt = np.tile(Xn, (1000, 1))
Yt = np.tile(Y, (1000, 1))
print("複製後的形狀", Xt.shape, Yt.shape)

trained = build_model()
trained.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
)
history = trained.fit(Xt, Yt, epochs=10, verbose=0)
print("每個 epoch 的損失", [round(float(v), 3) for v in history.history["loss"]])
print("訓練完的好烘機會", trained.predict(norm_l(X_try), verbose=0).reshape(-1).round(3))
print("訓練完第一層的 W1")
print(trained.get_layer("layer1").get_weights()[0].round(2))

plt.plot(history.history["loss"], marker="o")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss should fall")
plt.show()

### 1.8 每個單元管哪一塊

1.7 訓練出來的 `W1` 和存好的不一樣。三個單元沒有固定座位：這一次 1 號學到「太冷」，下一次可能改學「時間太短」。所以這一格回到存好的權重，三張圖每次打開都一樣。

每張圖畫第一層一個單元在整片溫度、時間上的輸出，越藍越接近 1。

In [ ]:
def unit_grid(unit):
    temperature = np.linspace(150, 285, 40)
    duration = np.linspace(11.5, 15.5, 40)
    tt, dd = np.meshgrid(temperature, duration)
    raw = np.column_stack([tt.ravel(), dd.ravel()])
    xn_grid = np.asarray(norm_l(raw))
    prob = dense(xn_grid, W1[:, unit], b1[unit])
    return tt, dd, prob.reshape(tt.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for unit, ax in enumerate(axes):
    tt, dd, prob = unit_grid(unit)
    shade = ax.pcolormesh(tt, dd, prob, cmap="Blues", vmin=0, vmax=1, shading="nearest")
    ax.scatter(X[good, 0], X[good, 1], marker="x", c="red", s=30)
    ax.scatter(X[~good, 0], X[~good, 1], marker="o", facecolors="none", edgecolors="#0D5BDC", s=30)
    draw_good_region(ax)
    ax.set_title(f"layer 1, unit {unit}")
    ax.set_xlabel("Temperature (Celsius)")
fig.colorbar(shade, ax=axes.ravel().tolist())
axes[0].set_ylabel("Duration (minutes)")
plt.show()

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">三個單元裡，你看得出哪一個在管溫度太低、哪一個在管時間嗎：</li>
<li style="color:#1a1a1a;">第二層的 <code style="color:#1a1a1a;">W2</code> 三個數都是負的。單元輸出越大，好烘機會會變大還變小：</li>
</ol>

</div>

### 1.9 實驗：改門檻

下一格只改門檻，網路用的是 1.6 放進存好權重的 `model`。模型輸出的是「好烘」的機會，門檻放低，更多爐被判成好烘；放高則相反。

In [ ]:
# 實驗：改 threshold，看判成好烘的範圍變大還變小
threshold = 0.5

test_prob = model.predict(norm_l(X_try), verbose=0).reshape(-1)
test_decision = (test_prob >= threshold).astype(int)
for row, prob, decision in zip(X_try, test_prob, test_decision):
    label = "好烘" if decision == 1 else "不好"
    print(f"{row[0]:.0f}°C、{row[1]} 分鐘 → 機會 {prob:.3f} → {label}")

temperature = np.linspace(150, 285, 50)
duration = np.linspace(11.5, 15.5, 50)
tt, dd = np.meshgrid(temperature, duration)
raw = np.column_stack([tt.ravel(), dd.ravel()])
grid_prob = model.predict(norm_l(raw), verbose=0).reshape(tt.shape)

fig, ax = plt.subplots()
ax.contourf(tt, dd, grid_prob >= threshold, levels=[-0.1, 0.5, 1.1], colors=["#d6e6ff", "#ffd6d6"], alpha=0.8)
ax.scatter(X[good, 0], X[good, 1], marker="x", c="red", s=40, label="actual good")
ax.scatter(X[~good, 0], X[~good, 1], marker="o", facecolors="none", edgecolors="#0D5BDC", s=40, label="actual bad")
draw_good_region(ax)
ax.set_title(f"threshold {threshold}")
ax.set_xlabel("Temperature (Celsius)")
ax.set_ylabel("Duration (minutes)")
ax.legend(loc="upper right")
plt.show()

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">門檻從 0.5 調高之後，<code style="color:#1a1a1a;">17 分鐘</code> 那一筆還是不好。<code style="color:#1a1a1a;">13.9 分鐘</code> 那一筆呢：</li>
<li style="color:#1a1a1a;">圖上判成好烘的顏色區域變大還變小。它有沒有蓋過紫線外面：</li>
<li style="color:#1a1a1a;">若你是烘焙師，你會把門檻放高還是放低：</li>
</ol>

</div>

實驗本到這裡。接下來用瀏覽器打開同資料夾的 `委託.html`，在 `體能等級.ipynb` 裡做體能等級估計。